# KTO Fine-Tuning: Meta-Llama-3-8B-Instruct

**KTO (Kahneman-Tversky Optimization)** is a preference alignment technique inspired by prospect theory that works with binary feedback (desirable/undesirable) rather than paired comparisons.

**Key Advantages of KTO over ORPO/DPO:**
- Works with unpaired data - doesn't require matched chosen/rejected pairs
- More data-efficient: can use any feedback data labeled as good or bad
- Based on prospect theory: models human loss aversion
- Often performs comparably or better with less structured data

**Dataset Format Required:**
- `prompt`: The user question/instruction
- `completion`: The response (can be either good or bad)
- `label`: Boolean - True for desirable, False for undesirable

**Note:** Our dataset is already in KTO format with `prompt`, `completion`, and `label` columns.

## 1. Environment Setup & Dependencies

In [1]:
# ============================================================================
# STEP 1: Install all required packages (KERNEL-SAFE VERSION)
# ============================================================================
# Note: KTO requires trl>=0.8.0 for KTOTrainer support

# Downgrade NumPy to prevent compatibility issues
!python -m pip install "numpy<2.0" --upgrade -q > /dev/null 2>&1

# Ensure clean pip and build tools
!python -m pip install pip==23.3.1 --upgrade -q > /dev/null 2>&1
!python -m pip install wheel setuptools --upgrade -q > /dev/null 2>&1

# Install PyTorch 2.1.0 with CUDA 11.8
!python -m pip install \
    torch==2.1.0+cu118 \
    torchvision==0.16.0+cu118 \
    torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu118 \
    -q > /dev/null 2>&1

# Install ML / training packages
# IMPORTANT: trl>=0.8.0 is required for KTOTrainer
!python -m pip install -q \
    transformers==4.42.0 \
    peft==0.11.1 \
    accelerate==0.31.0 \
    datasets==2.19.0 \
    scipy==1.12.0 \
    trl==0.9.4 \
    hf_transfer==0.1.5 \
    wandb==0.16.3 \
    matplotlib

!pip install rich

# Install flash-attention for memory-efficient attention
!python -m pip install flash-attn==2.5.5 --no-build-isolation -q 


[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.0/310.0 kB 8.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 30.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# from datasets import Dataset, load_dataset
# import pandas as pd

# # Option 1: Load directly from CSV
# dataset = load_dataset("csv", data_files="human_alignment_kto.csv")
# print(dataset)

# # Option 2: Load via pandas (more control)
# df = pd.read_csv("human_alignment_kto.csv")
# dataset = Dataset.from_pandas(df)
# print(dataset)

In [3]:
# from huggingface_hub import login
# login(token=os.environ['HF_TOKEN'])

# dataset.push_to_hub("agi123/optima_restore_ha_kto")

## 2. Authentication & Logging Setup

In [4]:
# ============================================================================
# STEP 2: Import libraries and authenticate with HuggingFace & Weights & Biases
# ============================================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os
import wandb

# Login to HuggingFace Hub (required for gated models like Llama)
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

# Initialize Weights & Biases for experiment tracking
wandb.login(key=os.environ['WANDB_API_KEY'])
run = wandb.init(
    project='kto_finetune_Meta-Llama-3-8B',  # Project name for KTO
    job_type='training',
    anonymous='allow'
)

/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
wandb: Currently logged in as: radhakrishna5a99 (radhakrishna5a99-innomatics-research-labs). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api ke

In [5]:
# ============================================================================
# STEP 3: Import all required modules for KTO training
# ============================================================================

from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TextStreamer
from trl import KTOConfig, KTOTrainer  # KTO-specific imports
import torch
import gc

## 3. Model Configuration

In [6]:
# ============================================================================
# STEP 4: Define model and cache paths
# ============================================================================

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"  # Base model to fine-tune
cache_dir = "./model-cache"  # Local cache for downloaded model weights

In [7]:
# ============================================================================
# STEP 5: Device map generator for multi-GPU distribution
# ============================================================================
# This function distributes model layers across available GPUs evenly
# Useful when model is too large for single GPU

def generate_device_map(num_layers: int, num_gpus: int) -> dict:
    """
    Generate a device map to distribute model layers across GPUs.
    
    Args:
        num_layers: Number of transformer layers in the model (32 for Llama-3-8B)
        num_gpus: Number of GPUs available
    
    Returns:
        dict: Mapping of layer names to GPU indices
    """
    device_map = {}
    layers_per_gpu = num_layers // num_gpus
    remainder = num_layers % num_gpus

    layer_id = 0
    for gpu in range(num_gpus):
        # Distribute remainder layers to first GPUs
        this_gpu_layer_count = layers_per_gpu + (1 if gpu < remainder else 0)
        for _ in range(this_gpu_layer_count):
            device_map[f"model.layers.{layer_id}"] = gpu
            layer_id += 1

    # Place embedding and output layers
    device_map["model.embed_tokens"] = 0          # Input embeddings on first GPU
    device_map["model.norm"] = num_gpus - 1       # Final norm on last GPU
    device_map["lm_head"] = num_gpus - 1          # Output head on last GPU

    return device_map

## 4. Load Base Model

In [8]:
# ============================================================================
# STEP 6: Load the base model
# ============================================================================
# Using bfloat16 for memory efficiency on Ampere+ GPUs
# Flash Attention 2 reduces memory usage and speeds up training

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map=generate_device_map(32, 1),  # 32 layers for Llama-3-8B, 1 GPU
    torch_dtype=torch.bfloat16,             # Memory-efficient dtype for Ampere GPUs
    attn_implementation="flash_attention_2", # Faster & less memory
    cache_dir=cache_dir
)

# Load tokenizer with fast implementation
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    trust_remote_code=True
)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [9]:
# ============================================================================
# STEP 7: Test base model before KTO training
# ============================================================================
# This shows the model's behavior BEFORE preference optimization

test_question = "According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?"

# Format as chat message
messages = [{"role": "user", "content": test_question}]
input_ids = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    return_tensors="pt", 
    add_generation_prompt=True
).to(model.device)

# Generate response
print("="*50)
print("BASE MODEL RESPONSE (before KTO training):")
print("="*50)

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASE MODEL RESPONSE (before KTO training):
user

According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?assistant

According to HDFC ERGO Optima Restore policy, you can get a second opinion from another doctor, and it can be covered by insurance under certain conditions.

Here are the details:

1. **Second Medical Opinion**: The policy allows you to get a second medical opinion from a specialist of your choice, if you are not satisfied with the initial diagnosis or treatment plan.
2. **Coverage**: The second medical opinion is covered under the policy, subject to the following conditions:
	* The second opinion must be obtained from a specialist in the same or related field as the initial doctor.
	* The second opinion must be obtained within 30 days of the initial diagnosis.
	* The second opinion must be obtained from a doctor who is a member of the same or a similar medical ass

## 5. Apply LoRA Configuration

In [10]:
# ============================================================================
# STEP 8: Helper function to count trainable parameters
# ============================================================================

def print_trainable_parameters(model):
    """
    Print summary of trainable vs frozen parameters.
    Useful for verifying LoRA is applied correctly.
    """
    trainable_params = 0
    non_trainable_params = 0
    all_params = 0

    for name, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
        else:
            non_trainable_params += param.numel()

    print(f"\n{'='*50}")
    print(f"Parameter Summary:")
    print(f"{'='*50}")
    print(f"  Trainable params:     {trainable_params:,}")
    print(f"  Non-trainable params: {non_trainable_params:,}")
    print(f"  Total params:         {all_params:,}")
    print(f"  Trainable %:          {100 * trainable_params / all_params:.4f}%")

print("BEFORE LoRA:")
print_trainable_parameters(model)

BEFORE LoRA:

Parameter Summary:
  Trainable params:     8,030,261,248
  Non-trainable params: 0
  Total params:         8,030,261,248
  Trainable %:          100.0000%


In [11]:
# ============================================================================
# STEP 9: Configure LoRA (Low-Rank Adaptation)
# ============================================================================
# LoRA adds small trainable matrices to attention/MLP layers
# This dramatically reduces memory usage and training time

peft_config = LoraConfig(
    r=16,                    # Rank of LoRA matrices (higher = more capacity, more memory)
    lora_alpha=32,           # Scaling factor (typically 2x rank)
    target_modules=[         # Which layers to apply LoRA to
        "q_proj",            # Query projection in attention
        "k_proj",            # Key projection in attention
        "v_proj",            # Value projection in attention  
        "o_proj",            # Output projection in attention
        "gate_proj",         # MLP gate projection
        "up_proj",           # MLP up projection
        "down_proj",         # MLP down projection
    ],
    lora_dropout=0.05,       # Dropout for regularization
    bias="none",             # Don't train bias terms
    task_type="CAUSAL_LM",   # Task type for causal language modeling
)

# Apply LoRA to the model
model = get_peft_model(model, peft_config)

print("AFTER LoRA:")
print_trainable_parameters(model)

AFTER LoRA:

Parameter Summary:
  Trainable params:     41,943,040
  Non-trainable params: 8,030,261,248
  Total params:         8,072,204,288
  Trainable %:          0.5196%


## 6. Tokenizer Configuration

In [12]:
# ============================================================================
# STEP 10: Configure padding token
# ============================================================================
# Llama models don't have a default pad token
# We need to set one for batch training

if '<pad>' in tokenizer.get_vocab():
    print('Using <pad> token for padding')
    tokenizer.pad_token = '<pad>'
elif '<|pad|>' in tokenizer.get_vocab():
    print('Using <|pad|> token for padding')
    tokenizer.pad_token = '<|pad|>'
elif '<unk>' in tokenizer.get_vocab():
    print('Using <unk> token for padding')
    tokenizer.pad_token = '<unk>'
else:
    # Fallback: use EOS token (common for Llama-3)
    print(f'Using EOS token ({tokenizer.eos_token}) for padding')
    print('WARNING: This may not be ideal for chat fine-tuning')
    tokenizer.pad_token = tokenizer.eos_token

# Set padding side to right for causal LM
tokenizer.padding_side = "right"

Using EOS token (<|eot_id|>) for padding


## 7. Load & Prepare Dataset for KTO

**KTO Dataset Format:**
KTO works with binary labeled data:
- `prompt`: The question/instruction
- `completion`: The response (either good or bad)
- `label`: Boolean - `True` for desirable (chosen), `False` for undesirable (rejected)

Our dataset is already in this format, so we just need to load and split it.

In [13]:
# ============================================================================
# STEP 11: Load the KTO-format preference dataset
# ============================================================================

# Dataset already in KTO format with prompt, completion, label columns
dataset_name = "agi123/optima_restore_ha_kto"
data = load_dataset(dataset_name)

print(f"Dataset: {dataset_name}")
print(f"Columns: {data['train'].column_names}")
print(f"Total samples: {len(data['train'])}")

# Validate KTO format
assert 'prompt' in data['train'].column_names, "Missing 'prompt' column"
assert 'completion' in data['train'].column_names, "Missing 'completion' column"
assert 'label' in data['train'].column_names, "Missing 'label' column"
print("\n✓ Dataset is in valid KTO format!")

Generating train split:   0%|          | 0/4230 [00:00<?, ? examples/s]

Dataset: agi123/optima_restore_ha_kto
Columns: ['prompt', 'completion', 'label']
Total samples: 4230

✓ Dataset is in valid KTO format!


In [14]:
# ============================================================================
# STEP 12: Verify dataset and show statistics
# ============================================================================
# Dataset is already in KTO format, just assign and show stats

kto_dataset = data['train']

# Count desirable vs undesirable samples
desirable_count = sum(1 for label in kto_dataset['label'] if label == True)
undesirable_count = len(kto_dataset) - desirable_count

print(f"KTO Dataset Statistics:")
print(f"{'='*50}")
print(f"Total samples: {len(kto_dataset)}")
print(f"  - Desirable (label=True): {desirable_count} ({100*desirable_count/len(kto_dataset):.1f}%)")
print(f"  - Undesirable (label=False): {undesirable_count} ({100*undesirable_count/len(kto_dataset):.1f}%)")

# Show sample
print(f"\nSample entry:")
print(kto_dataset[0])

KTO Dataset Statistics:
Total samples: 4230
  - Desirable (label=True): 2115 (50.0%)
  - Undesirable (label=False): 2115 (50.0%)

Sample entry:
{'prompt': 'Is the cost of a splint covered under the HDFC ERGO Optima Restore policy?', 'completion': 'No, the cost of a splint is not covered under the HDFC ERGO Optima Restore policy.', 'label': True}


In [15]:
# ============================================================================
# STEP 13: Shuffle the dataset
# ============================================================================
# Important: Shuffle so that desirable/undesirable examples are interleaved
# This helps with training stability

kto_dataset = kto_dataset.shuffle(seed=42)

print("Dataset shuffled!")
print(f"First 5 labels after shuffle: {kto_dataset['label'][:5]}")

Dataset shuffled!
First 5 labels after shuffle: [True, True, False, True, False]


In [16]:
# ============================================================================
# STEP 14: Split into train and evaluation sets
# ============================================================================
# We'll use 95% for training and 5% for evaluation

split_dataset = kto_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

# Check label distribution in train set
train_desirable = sum(train_dataset['label'])
train_undesirable = len(train_dataset) - train_desirable
print(f"\nTraining set distribution:")
print(f"  - Desirable: {train_desirable} ({100*train_desirable/len(train_dataset):.1f}%)")
print(f"  - Undesirable: {train_undesirable} ({100*train_undesirable/len(train_dataset):.1f}%)")

Training samples: 4018
Evaluation samples: 212

Training set distribution:
  - Desirable: 2018 (50.2%)
  - Undesirable: 2000 (49.8%)


In [17]:
# ============================================================================
# STEP 15: Inspect samples to verify formatting
# ============================================================================

print("Sample DESIRABLE example:")
print("=" * 50)
for i, sample in enumerate(train_dataset):
    if sample['label'] == True:
        print(f"PROMPT: {sample['prompt'][:200]}...")
        print(f"\nCOMPLETION: {sample['completion'][:200]}...")
        print(f"\nLABEL: {sample['label']} (Desirable)")
        break

print("\n" + "=" * 50)
print("Sample UNDESIRABLE example:")
print("=" * 50)
for i, sample in enumerate(train_dataset):
    if sample['label'] == False:
        print(f"PROMPT: {sample['prompt'][:200]}...")
        print(f"\nCOMPLETION: {sample['completion'][:200]}...")
        print(f"\nLABEL: {sample['label']} (Undesirable)")
        break

Sample DESIRABLE example:
PROMPT: What is the policy code for HDFC ERGO Optima Restore?...

COMPLETION: The policy code for HDFC ERGO Optima Restore is HDFHLIP205V12/0425....

LABEL: True (Desirable)

Sample UNDESIRABLE example:
PROMPT: What is the refund policy for yearly payment options under the HDFC ERGO Optima Restore policy?...

COMPLETION: For yearly payment options under the HDFC ERGO Optima Restore policy, the cancellation grid as per 5-Year Tenure policies will be applicable....

LABEL: False (Undesirable)


## 8. Training Configuration

In [18]:
# ============================================================================
# STEP 16: Define training hyperparameters
# ============================================================================

# Model naming
model_name = model_id.split("/")[-1]
dataset_short_name = dataset_name.split("/")[-1]

# Training hyperparameters
epochs = 6                    # Number of training epochs
batch_size = 2                # Per-device batch size (reduce if OOM)
grad_accum = 4                # Gradient accumulation steps (effective batch = batch_size * grad_accum)
learning_rate = 5e-6          # Learning rate (lower than SFT, typical for preference learning)
max_length = 1024             # Maximum sequence length for prompt + response
max_prompt_length = 512       # Maximum length for just the prompt

# KTO-specific hyperparameters
beta = 0.1                    # KTO beta - controls the strength of the KL penalty
                              # Higher = stronger preference signal
                              # Typical range: 0.01 - 0.5

desirable_weight = 1.0        # Weight for desirable (chosen) examples
undesirable_weight = 1.0      # Weight for undesirable (rejected) examples
                              # Adjust if you have imbalanced data

# Regularization
warmup_ratio = 0.1            # Warmup steps as ratio of total steps

# Output paths
fine_tune_tag = 'kto_health_insurance'
save_dir = f'./results/{model_name}_{dataset_short_name}_{epochs}_epochs_kto-{fine_tune_tag}'

print(f"Output directory: {save_dir}")
print(f"Effective batch size: {batch_size * grad_accum}")

Output directory: ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_insurance
Effective batch size: 8


In [19]:
# ============================================================================
# STEP 17: Custom logging callback for detailed tracking
# ============================================================================

import transformers
import os

class KTOLoggingCallback(transformers.TrainerCallback):
    """
    Custom callback to log KTO-specific metrics to a file.
    KTO logs metrics like:
    - kto_loss: The KTO-specific loss
    - rewards/chosen: Reward for desirable responses
    - rewards/rejected: Reward for undesirable responses
    - rewards/margins: Difference between chosen and rejected rewards
    """
    def __init__(self, log_file_path, save_dir):
        self.log_file_path = log_file_path
        self.save_dir = save_dir

    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        with open(self.log_file_path, 'a') as f:
            # Log training loss
            if 'loss' in logs:
                f.write(f"Step: {state.global_step}, Training Loss: {logs['loss']:.4f}\n")
            
            # Log eval loss
            if 'eval_loss' in logs:
                f.write(f"Step: {state.global_step}, Eval Loss: {logs['eval_loss']:.4f}\n")
            
            # Log KTO-specific metrics
            if 'rewards/margins' in logs:
                f.write(f"Step: {state.global_step}, Reward Margin: {logs['rewards/margins']:.4f}\n")
            
            if 'kl' in logs:
                f.write(f"Step: {state.global_step}, KL Divergence: {logs['kl']:.4f}\n")
            
            f.flush()

        # Save checkpoint with trainable params
        if state.global_step % int(args.save_steps) == 0 and state.global_step > 0:
            checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
            os.makedirs(checkpoint_dir, exist_ok=True)
            
            # Save trainable parameters
            current_trainable_params = {n: p.data for n, p in model.named_parameters() if p.requires_grad}
            file_path = os.path.join(checkpoint_dir, "trainable_params.bin")
            torch.save(current_trainable_params, file_path)

# Initialize callback
os.makedirs(cache_dir, exist_ok=True)
log_file_path = os.path.join(cache_dir, "kto_training_logs.txt")
logging_callback = KTOLoggingCallback(log_file_path, save_dir)

print(f"Logging to: {log_file_path}")

Logging to: ./model-cache/kto_training_logs.txt


## 9. Initialize KTO Trainer

In [20]:
# ============================================================================
# STEP 18: Configure KTOTrainer
# ============================================================================
# KTO uses prospect theory to model human preferences
# It applies different weights to gains (desirable) vs losses (undesirable)

kto_config = KTOConfig(
    # Output configuration
    output_dir=save_dir,
    
    # Training duration
    num_train_epochs=epochs,
    # max_steps=10,  # Uncomment for quick test run
    
    # Batch size configuration
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    
    # Learning rate configuration
    learning_rate=learning_rate,
    lr_scheduler_type="cosine",       # Cosine decay schedule
    warmup_ratio=warmup_ratio,         # Gradual warmup at start
    
    # KTO-specific parameters
    beta=beta,                         # KL penalty strength
    desirable_weight=desirable_weight,  # Weight for desirable examples
    undesirable_weight=undesirable_weight,  # Weight for undesirable examples
    max_length=max_length,             # Max total sequence length
    max_prompt_length=max_prompt_length,  # Max prompt length
    
    # Precision and optimization
    bf16=True,                         # Use bfloat16 for Ampere+ GPUs
    # fp16=True,                       # Use for older GPUs (V100, etc.)
    optim="adamw_torch",               # Optimizer
    max_grad_norm=0.3,                 # Gradient clipping
    
    # Evaluation configuration
    eval_strategy="steps",
    eval_steps=0.1,                    # Evaluate every 10% of training
    do_eval=True,
    
    # Logging configuration
    logging_steps=1,                   # Log every step
    log_level="info",
    report_to="wandb",                 # Log to Weights & Biases
    run_name=f"kto-{model_name}-{fine_tune_tag}",
    
    # Checkpointing
    save_strategy="steps",
    save_steps=200,                    # Save checkpoint every 200 steps
    save_total_limit=3,                # Keep only 3 most recent checkpoints
    
    # Hub configuration (optional)
    hub_private_repo=True,
    
    # Reproducibility
    seed=42,
)

In [21]:
# ============================================================================
# STEP 19: Verify dataset format for KTOTrainer
# ============================================================================

print("Dataset sample:")
print(train_dataset[0])
print()
print(f"Type of prompt: {type(train_dataset[0]['prompt'])}")
print(f"Type of completion: {type(train_dataset[0]['completion'])}")
print(f"Type of label: {type(train_dataset[0]['label'])}")

Dataset sample:
{'prompt': 'What is the policy code for HDFC ERGO Optima Restore?', 'completion': 'The policy code for HDFC ERGO Optima Restore is HDFHLIP205V12/0425.', 'label': True}

Type of prompt: <class 'str'>
Type of completion: <class 'str'>
Type of label: <class 'bool'>


In [22]:
# ============================================================================
# STEP 20: Initialize the KTOTrainer
# ============================================================================

trainer = KTOTrainer(
    model=model,
    args=kto_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    callbacks=[logging_callback],
)

print("\nKTOTrainer initialized successfully!")
print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print(f"Beta (KL penalty): {beta}")
print(f"Desirable weight: {desirable_weight}")
print(f"Undesirable weight: {undesirable_weight}")

/usr/local/lib/python3.11/dist-packages/trl/trainer/kto_trainer.py:504: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your KTOConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/4018 [00:00<?, ? examples/s]

Extracting KL train dataset:   0%|          | 0/4018 [00:00<?, ? examples/s]

Processing tokenized train dataset:   0%|          | 0/4018 [00:00<?, ? examples/s]

Processing tokenized train KL dataset:   0%|          | 0/4018 [00:00<?, ? examples/s]

...

Processing tokenized eval KL dataset:   0%|          | 0/212 [00:00<?, ? examples/s]

Filtering desirable examples:   0%|          | 0/4018 [00:00<?, ? examples/s]

Filtering undesirable examples:   0%|          | 0/4018 [00:00<?, ? examples/s]

Using auto half precision backend



KTOTrainer initialized successfully!
Training samples: 4018
Evaluation samples: 212
Beta (KL penalty): 0.1
Desirable weight: 1.0
Undesirable weight: 1.0


## 10. Train the Model

In [23]:
# ============================================================================
# STEP 21: Start KTO training
# ============================================================================

print("\n" + "="*50)
print("Starting KTO Training...")
print("="*50)

trainer.train()


Starting KTO Training...


***** Running training *****
  Num examples = 4,018
  Num Epochs = 6
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 3,012
  Number of trainable parameters = 41,943,040
Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss,Rewards/chosen,Logps/chosen,Rewards/rejected,Logps/rejected,Rewards/margins,Kl
302,0.424000,0.374280,0.169495,-75.446490,-0.886617,-89.495168,1.056111,0.185294
604,0.121900,0.188854,1.467801,-62.463434,-2.549641,-106.125416,4.017442,0.282968
906,0.133700,0.144188,1.818241,-58.959020,-3.589927,-116.528278,5.408168,0.116427
1208,0.155100,0.121717,1.968461,-57.456820,-4.073206,-121.361065,6.041667,0.094960
1510,0.081100,0.113052,2.058257,-56.558866,-4.284769,-123.476681,6.343026,0.073857
1812,0.206000,0.109669,2.043042,-56.711018,-4.528969,-125.918699,6.572011,0.048741
2114,0.157700,0.108514,2.072597,-56.415467,-4.494014,-125.569149,6.566612,0.050944
2416,0.060800,0.108210,2.077870,-56.362728,-4.499437,-125.623378,6.577307,0.057755
2718,0.073700,0.108156,2.078771,-56.353732,-4.500525,-125.634265,6.579296,0.047300


Saving model checkpoint to ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_insurance/checkpoint-200


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.42.0",
  "use_cache": true,
  "vocab_size": 128256
}

tokenizer config file saved in ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_in

TrainOutput(global_step=3012, training_loss=0.1663360921282613, metrics={'train_runtime': 5135.0563, 'train_samples_per_second': 4.695, 'train_steps_per_second': 0.587, 'total_flos': 0.0, 'train_loss': 0.1663360921282613, 'epoch': 5.99701343952215})

## 11. Save the Fine-tuned Model

In [24]:
# ============================================================================
# STEP 22: Save the fine-tuned model
# ============================================================================

print(f"\nSaving model to: {save_dir}")

# Save the model
trainer.save_model(save_dir)

# Save the tokenizer
tokenizer.save_pretrained(save_dir)

print("Model and tokenizer saved!")

Saving model checkpoint to ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_insurance



Saving model to: ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_insurance


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.42.0",
  "use_cache": true,
  "vocab_size": 128256
}

tokenizer config file saved in ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_in

Model and tokenizer saved!


In [25]:
# ============================================================================
# VERIFY: Check if we have the trained model
# ============================================================================

# Check if model and trainer.model are the same object
print(f"Are model and trainer.model the same object? {model is trainer.model}")

# Quick test to verify trained behavior
test_q = "What is the Restore Benefit in HDFC ERGO Optima Restore?"
messages = [{"role": "user", "content": test_q}]
input_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(input_ids, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)

print(f"\nResponse from trainer.model:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Check adapter weights are non-zero (indicating training happened)
for name, param in trainer.model.named_parameters():
    if 'lora' in name.lower() and param.requires_grad:
        print(f"\n{name}: mean={param.data.mean():.6f}, std={param.data.std():.6f}")
        break

Are model and trainer.model the same object? False


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:545: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



Response from trainer.model:
user

What is the Restore Benefit in HDFC ERGO Optima Restore?assistant

In HDFC ERGO Optima Restore, the Restore Benefit is a feature that allows the policy to restore the basic sum insured in case the basic sum insured is fully utilized during a policy year. This means that if the insured person exhausts the entire basic sum insured, the basic sum insured will be restored to its full value, allowing them to continue receiving coverage for future medical expenses.

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: mean=0.000046, std=0.009033


In [26]:
# ============================================================================
# STEP 22b: Push the fine-tuned model to Hugging Face Hub
# ============================================================================

adapter_model = f"agi123/{model_name}-kto-{fine_tune_tag}-effective_bactch_size-8-epochs-{epochs}-adapters"

print(f"Pushing model to HuggingFace Hub: {adapter_model}")

# Push the TRAINED model (use trainer.model, not model!)
trainer.model.push_to_hub(
    adapter_model,
    use_auth_token=True,
    max_shard_size="10GB",
    use_safetensors=True
)

# Push tokenizer
tokenizer.push_to_hub(adapter_model, use_auth_token=True)

print(f"\n✓ Model pushed to: https://huggingface.co/{adapter_model}")

Pushing model to HuggingFace Hub: agi123/Meta-Llama-3-8B-Instruct-kto-kto_health_insurance-effective_bactch_size-8-epochs-6-adapters


/usr/local/lib/python3.11/dist-packages/transformers/utils/hub.py:875: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

tokenizer config file saved in /tmp/tmpe2e6h9ll/tokenizer_config.json
Special tokens file saved in /tmp/tmpe2e6h9ll/special_tokens_map.json
Uploading the following files to agi123/Meta-Llama-3-8B-Instruct-kto-kto_health_insurance-effective_bactch_size-8-epochs-6-adapters: tokenizer_config.json,special_tokens_map.json,tokenizer.json,README.md



✓ Model pushed to: https://huggingface.co/agi123/Meta-Llama-3-8B-Instruct-kto-kto_health_insurance-effective_bactch_size-8-epochs-6-adapters


In [ ]:
https://huggingface.co/agi123/Meta-Llama-3-8B-Instruct-kto-kto_health_insurance-effective_bactch_size-8-epochs-6-adapters

## 12. Test the Fine-tuned Model

In [27]:
# ============================================================================
# STEP 23: Test the fine-tuned model
# ============================================================================

test_question = "According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?"

# Format as chat message
messages = [{"role": "user", "content": test_question}]
input_ids = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    return_tensors="pt", 
    add_generation_prompt=True
).to(model.device)

# Generate response
print("="*50)
print("FINE-TUNED MODEL RESPONSE (after KTO training):")
print("="*50)

with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

FINE-TUNED MODEL RESPONSE (after KTO training):
user

According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?assistant

According to HDFC ERGO Optima Restore, if you are diagnosed with a Critical Illness, you can get a second opinion from another doctor, but it will not be covered by insurance. The policy specifically states that the diagnosis and treatment by the Network Hospital or the treating doctor shall be considered as the final diagnosis and treatment, and any additional or alternative diagnosis or treatment shall not be admissible for claim purposes.


In [28]:
# ============================================================================
# STEP 24: Test with multiple questions
# ============================================================================

test_questions = [
    "What is the waiting period for pre-existing diseases in HDFC ERGO Optima Restore?",
    "Does HDFC ERGO Optima Restore cover maternity expenses?",
    "What is the Restore Benefit in HDFC ERGO Optima Restore?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"Question {i}: {question}")
    print("="*50)
    
    messages = [{"role": "user", "content": question}]
    input_ids = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        return_tensors="pt", 
        add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    print(f"\nResponse: {response}")


Question 1: What is the waiting period for pre-existing diseases in HDFC ERGO Optima Restore?

Response: The waiting period for pre-existing diseases in HDFC ERGO Optima Restore is 24 months.

Question 2: Does HDFC ERGO Optima Restore cover maternity expenses?

Response: According to the HDFC ERGO Optima Restore policy, maternity expenses are not covered.

Question 3: What is the Restore Benefit in HDFC ERGO Optima Restore?

Response: In HDFC ERGO Optima Restore, the Restore Benefit is a feature that allows the policy to restore the basic sum insured in case the basic sum insured is fully utilized during a policy year. This means that if the insured person exhausts the entire basic sum insured, the basic sum insured will be restored to its full value, allowing them to continue receiving coverage for future medical expenses.


## 13. Log Artifacts to W&B

In [29]:
# ============================================================================
# STEP 25: Log training artifacts to Weights & Biases
# ============================================================================

import json

artifact_name = f'kto_run_{model_name}_{fine_tune_tag}_ebs_8_epochs_{epochs}'

# Create artifact directory
os.makedirs(f"artifacts/{artifact_name}", exist_ok=True)

# Save training arguments
training_args_dict = trainer.args.to_dict()
with open(f"artifacts/{artifact_name}/training_args.json", "w") as f:
    json.dump(training_args_dict, f, indent=4)

# Save evaluation results
eval_results = trainer.evaluate()
with open(f"artifacts/{artifact_name}/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=4)

# Save wandb run summary
summary_dict = dict(wandb.run.summary)
# Filter out non-serializable items
summary_dict = {k: v for k, v in summary_dict.items() if isinstance(v, (int, float, str, bool, list, dict))}
with open(f"artifacts/{artifact_name}/run_summary.json", "w") as f:
    json.dump(summary_dict, f, indent=4, default=str)

# Log to W&B
artifact = wandb.Artifact(
    name=f"{artifact_name}_config_and_results",
    type="training-config",
    description=f"KTO training config and results for {model_name}",
)
artifact.add_dir(f"artifacts/{artifact_name}")
run.log_artifact(artifact)

# Finish the wandb run
run.finish()

print("\n" + "="*50)
print("KTO Training Complete!")
print("="*50)
print(f"\nFinal eval loss: {eval_results.get('eval_loss', 'N/A')}")
print(f"Model saved to: {save_dir}")
print(f"Artifacts logged to W&B")


***** Running Evaluation *****
  Num examples = 212
  Batch size = 2


wandb: Adding directory to artifact (./artifacts/kto_run_Meta-Llama-3-8B-Instruct_kto_health_insurance_ebs_8_epochs_6)... Done. 0.0s


eval/kl,▅█▃▂▂▁▁▁▁▁
eval/logps/chosen,▁▆▇███████
eval/logps/rejected,█▅▃▂▁▁▁▁▁▁
eval/loss,█▃▂▁▁▁▁▁▁▁
eval/rewards/chosen,▁▆▇███████
eval/rewards/margins,▁▅▇▇██████
eval/rewards/rejected,█▅▃▂▁▁▁▁▁▁
eval/runtime,▄█▅█▄▁▁▇▄▃
eval/samples_per_second,▅▁▄▁▅▇█▂▅▆
eval/steps_per_second,▅▁▄▁▅▇█▂▅▆
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███



KTO Training Complete!

Final eval loss: 0.10781042277812958
Model saved to: ./results/Meta-Llama-3-8B-Instruct_optima_restore_ha_kto_6_epochs_kto-kto_health_insurance
Artifacts logged to W&B


In [ ]:
https://wandb.ai/radhakrishna5a99-innomatics-research-labs/kto_finetune_Meta-Llama-3-8B/runs/896qv5ng?nw=nwuserradhakrishna5a99

In [ ]:
# ============================================================================
# STEP 26: Cleanup (optional)
# ============================================================================
# Free up GPU memory

# del model
# del trainer
# gc.collect()
# torch.cuda.empty_cache()
# print("GPU memory cleared!")

## Appendix: KTO vs ORPO Comparison

| Aspect | KTO | ORPO |
|--------|-----|------|
| **Data Format** | Binary labels (desirable/undesirable) | Paired comparisons (chosen/rejected) |
| **Data Flexibility** | Can use unpaired data | Requires paired data |
| **Theoretical Basis** | Prospect theory (loss aversion) | Odds ratio optimization |
| **Reference Model** | Required | Not required |
| **Memory Usage** | Higher (needs ref model) | Lower (no ref model) |
| **Training Stages** | Single stage | Single stage |
| **Loss Function** | Human utility maximization | Odds ratio + SFT |

### When to use KTO:
- When you have non-paired preference data (e.g., thumbs up/down ratings)
- When you want to model human loss aversion explicitly
- When collecting paired comparisons is expensive

### When to use ORPO:
- When you have well-paired preference data
- When GPU memory is limited (no reference model needed)
- When training speed is a priority